Since my research question is about daily electricity demand, I merged the data by date instead of the full timestamp. The time of day isn't relevant to my question as I care about how a whole day's weather relates to that day's demand, not how demand changes hour to hour. 


When aggregating to a daily level, I summarised each feature by the statistic most relevant to demand — taking the daily maximum for temperature and demand (since peaks drive grid stress) and the daily mean for humidity, wind, and pressure (which affect demand through sustained conditions rather than momentary extremes).

In [45]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [83]:
df = pd.read_csv('combined_demand_data.csv')                

df2 = pd.read_csv("weather_all.csv",low_memory=False) 

In [84]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1658965 entries, 0 to 1658964
Data columns (total 5 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   REGION          1658965 non-null  object 
 1   SETTLEMENTDATE  1658965 non-null  object 
 2   TOTALDEMAND     1658965 non-null  float64
 3   RRP             1658965 non-null  float64
 4   PERIODTYPE      1658965 non-null  object 
dtypes: float64(2), object(3)
memory usage: 63.3+ MB


In [85]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716682 entries, 0 to 1716681
Data columns (total 29 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   record_id                 object 
 1   station_number            int64  
 2   year_local                int64  
 3   month_local               int64  
 4   day_local                 int64  
 5   hour_local                int64  
 6   min_local                 int64  
 7   year_std                  int64  
 8   month_std                 int64  
 9   day_std                   int64  
 10  hour_std                  int64  
 11  min_std                   int64  
 12  precip_mm                 float64
 13  precip_quality            object 
 14  air_temp_C                float64
 15  air_temp_quality          object 
 16  wet_bulb_C                float64
 17  wet_bulb_quality          object 
 18  dew_point_C               float64
 19  dew_point_quality         object 
 20  humidity_pct            

In [86]:
df2['station_number'].unique()       # Distinct station numbers

array([94029, 86338, 23090, 66062, 86071, 40913])

In [87]:
df['REGION'].unique()                 # which regions are in the demand data set

array(['NSW1', 'VIC1', 'TAS1', 'SA1', 'QLD1'], dtype=object)

In [88]:
# Convert humidity to numeric (corrupted text values become NaN), then drop those rows
df2['humidity_pct'] = pd.to_numeric(df2['humidity_pct'], errors='coerce')
df2 = df2.dropna(subset=['humidity_pct'])

# STEP 1: Build matching 'date' columns in both datasets

# --- Demand dataset: convert SETTLEMENTDATE text to a daily date ---
df['datetime'] = pd.to_datetime(df['SETTLEMENTDATE'], format='mixed')
df['date'] = df['datetime'].dt.date

# --- Weather dataset: build a date from separate year/month/day columns, and then convert it to a daily date ---
df2['date'] = pd.to_datetime(
    df2[['year_local', 'month_local', 'day_local']]
    .rename(columns={'year_local': 'year', 'month_local': 'month', 'day_local': 'day'})
).dt.date

# STEP 2: Map weather stations to states for the weather dataset
station_to_state = {
    66062: 'NSW1',
    86338: 'VIC1',
    86071: 'VIC1',
    23090: 'SA1',
    40913: 'QLD1',
    94029: 'TAS1',
}
df2['state'] = df2['station_number'].map(station_to_state)

In [89]:
df.head()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE,datetime,date
0,NSW1,2000/09/01 00:30,8117.23667,36.72,TRADE,2000-09-01 00:30:00,2000-09-01
1,NSW1,2000/09/01 01:00,7799.70000,32.92,TRADE,2000-09-01 01:00:00,2000-09-01
2,NSW1,2000/09/01 01:30,7453.99000,27.84,TRADE,2000-09-01 01:30:00,2000-09-01
3,NSW1,2000/09/01 02:00,7095.55333,30.62,TRADE,2000-09-01 02:00:00,2000-09-01
4,NSW1,2000/09/01 02:30,6786.22167,33.53,TRADE,2000-09-01 02:30:00,2000-09-01


In [90]:
df2.head()

,record_id,station_number,year_local,month_local,day_local,hour_local,min_local,year_std,month_std,day_std,...,humidity_quality,mslp_hPa,mslp_quality,station_pressure_hPa,station_pressure_quality,aws_flag,end_marker,source_file,date,state
0,hm,94029,2000,8,5,12,20,2000,8,5,...,N,1016.1,N,1010.1,N,2.0,#,/Users/aaronlgs/Monash University/ADS1002/Proj...,2000-08-05,TAS1
1,hm,94029,2000,8,15,12,20,2000,8,15,...,N,1016.1,N,1010.1,N,2.0,#,/Users/aaronlgs/Monash University/ADS1002/Proj...,2000-08-15,TAS1
2,hm,94029,2000,8,31,16,10,2000,8,31,...,N,998.1,N,992.0,N,1.0,#,/Users/aaronlgs/Monash University/ADS1002/Proj...,2000-08-31,TAS1
3,hm,94029,2000,8,31,16,12,2000,8,31,...,N,998.1,N,992.0,N,1.0,#,/Users/aaronlgs/Monash University/ADS1002/Proj...,2000-08-31,TAS1
4,hm,94029,2000,11,24,12,47,2000,11,24,...,N,1016.4,N,1010.2,N,1.0,#,/Users/aaronlgs/Monash University/ADS1002/Proj...,2000-11-24,TAS1


In [91]:
# STEP 3: Aggregate WEATHER to daily-per-state
weather_daily = df2.groupby(['state', 'date']).agg(
    # Temperature
    max_temp=('air_temp_C', 'max'),
    mean_temp=('air_temp_C', 'mean'),
    min_temp=('air_temp_C', 'min'),
    # Dew point (moisture content)
    max_dew=('dew_point_C', 'max'),
    mean_dew=('dew_point_C', 'mean'),
    min_dew=('dew_point_C', 'min'),
    # Wet bulb (combined heat + humidity)
    max_wet_bulb=('wet_bulb_C', 'max'),
    mean_wet_bulb=('wet_bulb_C', 'mean'),
    min_wet_bulb=('wet_bulb_C', 'min'),
    # Humidity
    max_humidity=('humidity_pct', 'max'),
    mean_humidity=('humidity_pct', 'mean'),
    min_humidity=('humidity_pct', 'min'),
    # Pressure
    max_pressure=('mslp_hPa', 'max'),
    mean_pressure=('mslp_hPa', 'mean'),
    min_pressure=('mslp_hPa', 'min'),
    # Rain
    total_rain=('precip_mm', 'max'),
).reset_index()

# STEP 4: Aggregate DEMAND to daily-per-state
demand_daily = df.groupby(['REGION', 'date']).agg(
    peak_demand=('TOTALDEMAND', 'max'),
    mean_demand=('TOTALDEMAND', 'mean'),
    mean_price=('RRP', 'mean'),
).reset_index()

In [92]:
weather_daily.sample()

,state,date,max_temp,mean_temp,min_temp,max_dew,mean_dew,min_dew,max_wet_bulb,mean_wet_bulb,min_wet_bulb,max_humidity,mean_humidity,min_humidity,max_pressure,mean_pressure,min_pressure,total_rain
13532,QLD1,2018-02-26,30.8,30.52,30.0,22.2,21.86,21.5,24.9,24.66,24.5,62.0,60.0,58.0,1006.8,1006.44,1006.0,0.0


In [93]:
demand_daily.sample()

,REGION,date,peak_demand,mean_demand,mean_price
10029,QLD1,2007-06-26,7305.99,6121.797083,326.557292


In [94]:
# STEP 5: Merge on state/region + date
model_data = pd.merge(
    weather_daily,
    demand_daily,
    left_on=['state', 'date'],
    right_on=['REGION', 'date'],
    how='inner'
)

# drop the duplicate REGION column
model_data = model_data.drop(columns='REGION')

print("Done. Shape:", model_data.shape)
print(model_data['state'].value_counts())
model_data.head()

Done. Shape: (33672, 21)
state
VIC1    7087
NSW1    7085
SA1     7085
QLD1    7083
TAS1    5332
Name: count, dtype: int64


,state,date,max_temp,mean_temp,min_temp,max_dew,mean_dew,min_dew,max_wet_bulb,mean_wet_bulb,...,max_humidity,mean_humidity,min_humidity,max_pressure,mean_pressure,min_pressure,total_rain,peak_demand,mean_demand,mean_price
0,NSW1,2000-07-24,22.9,19.118182,16.3,6.9,6.204545,5.0,14.5,12.677273,...,52.0,43.590909,33.0,1013.1,1012.454545,1011.6,0.0,10417.93167,8720.684653,43.315625
1,NSW1,2000-07-25,23.5,17.183333,11.5,10.8,7.427083,4.8,15.4,12.237500,...,88.0,54.604167,37.0,1013.2,1009.700000,1006.0,3.2,10073.98667,8595.720694,44.491042
2,NSW1,2000-07-26,17.1,13.229167,9.5,10.3,3.472917,-3.6,11.7,9.039583,...,84.0,55.416667,26.0,1010.3,1006.179167,1004.1,3.6,10754.84833,8693.146667,53.784792
3,NSW1,2000-07-27,15.3,9.812500,5.8,5.4,0.981250,-2.3,8.6,6.195833,...,95.0,56.500000,30.0,1018.6,1013.545833,1010.5,7.8,11506.91667,9303.345035,77.902917
4,NSW1,2000-07-28,16.0,10.641667,6.3,5.6,3.891667,1.0,11.0,7.625000,...,94.0,64.437500,46.0,1027.5,1022.779167,1018.2,14.6,10816.12833,9207.272396,62.201667


In [95]:
print("Weather_daily range:", weather_daily['date'].min(), "to", weather_daily['date'].max())
print("Demand_daily range:", demand_daily['date'].min(), "to", demand_daily['date'].max())
print("Weather_daily rows:", len(weather_daily))
print("Demand_daily rows:", len(demand_daily))

Weather_daily range: 2000-07-24 to 2020-01-20
Demand_daily range: 2000-01-01 to 2020-01-01
Weather_daily rows: 34900
Demand_daily rows: 34518


In [96]:
   print(weather_daily['date'].dtype)
   print(demand_daily['date'].dtype)
   print(weather_daily['date'].iloc[0], type(weather_daily['date'].iloc[0]))
   print(demand_daily['date'].iloc[0], type(demand_daily['date'].iloc[0]))

object
object
2000-07-24 <class 'datetime.date'>
2000-01-01 <class 'datetime.date'>


In [97]:
model_data.to_csv("new_model_data.csv", index=False)

In [98]:
print("Before merge - weather_daily:", len(weather_daily), "demand_daily:", len(demand_daily))
print("After merge - model_data:", len(model_data))

Before merge - weather_daily: 34900 demand_daily: 34518
After merge - model_data: 33672


## Features left out of mergeing 

### Quality-flag columns (metadata about data quality, not measurements):

precip_quality, air_temp_quality, wet_bulb_quality, dew_point_quality, humidity_quality, wind_speed_quality, wind_dir_quality, max_gust_quality, mslp_quality, station_pressure_quality — these are BoM's data-quality codes (e.g. "N" for normal). They describe how reliable each reading is, not the weather. They're useful for filtering bad data, but they aren't weather variables and have no relationship with demand. Excluded as features (though they could inform data-cleaning decisions).

### Actual weather measurements

wind_dir_deg — wind direction (0–360°). Excluded because direction is a circular variable — 359° and 1° are nearly the same direction, but numerically far apart, so it has no meaningful linear relationship with demand. It would need special (cyclical) encoding to use properly, and there's no strong physical reason wind direction (as opposed to speed) drives electricity demand. Low value, high complexity → excluded.


station_pressure_hPa — station-level pressure. Excluded because it's redundant with mslp_hPa (mean sea-level pressure), which you did include. The two measure essentially the same atmospheric pressure, just one adjusted for altitude. Including both would cause multicollinearity for no information gain. You picked mslp (the standard meteorological choice) and dropped this to avoid redundancy.


min_temp / raw temperature (if you used discomfort features instead) — excluded from the final model because you engineered heat_discomfort/cold_discomfort from temperature, and including raw temperature alongside them would be redundant (they're derived from it), causing multicollinearity.

In [99]:
model_data['state'].unique()

array(['NSW1', 'QLD1', 'SA1', 'TAS1', 'VIC1'], dtype=object)